# 09.9 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/09-calling-llms-from-python/solutions.ipynb)

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [2]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


## Exercise 1

Per request: 520 input tokens, 10 output tokens. Per day: 5,200,000 input and 100,000 output tokens.

- Haiku 5.5: 5.2 × $0.10 + 0.1 × $0.50 = $0.52 + $0.05 = **$0.57 a day**, about **$17 a month**.
- Opus 5.5: 5.2 × $4 + 0.1 × $20 = $20.80 + $2.00 = **$22.80 a day**, about **$684 a month**.

The system prompt is 77% of the input tokens. Shortening it, or caching it, is the first saving.

In [3]:
# US dollars per million tokens (input, output). Prices change: check the provider's
# pricing page before you rely on these. Claude prices as published in October 2026.
PRICES = {
    "Claude Haiku 5.5":  (0.10, 0.50),
    "Claude Sonnet 5.5": (2.00, 10.00),
    "Claude Opus 5.5":   (4.00, 20.00),
    "Ollama (local)":    (0.00, 0.00),     # you pay in hardware and electricity instead
}

def cost(n_requests, tokens_in, tokens_out, price_in, price_out):
    return n_requests * (tokens_in * price_in + tokens_out * price_out) / 1_000_000

In [4]:
for name in ["Claude Haiku 5.5", "Claude Opus 5.5"]:
    day = cost(10_000, 400 + 120, 10, *PRICES[name])
    print(f"{name:16} ${day:6.2f} a day, ${day * 30:7.2f} a month")

Claude Haiku 5.5 $  0.57 a day, $  17.10 a month
Claude Opus 5.5  $ 22.80 a day, $ 684.00 a month


## Exercise 2

In [5]:
import time
LOG = []

def logged_ask(prompt, **kwargs):
    start = time.time()
    answer = ask(prompt, **kwargs)
    LOG.append({"chars": len(prompt), "in": course_llm.last_usage["input"],
                "out": course_llm.last_usage["output"], "secs": round(time.time() - start, 2)})
    return answer

In [6]:
import pandas as pd
policy = repo_file("data/shopwave_docs/returns.md").read_text()
for prompt in ["Say hi.", "Summarise in one line: " + policy[:400],
               "Summarise in one line: " + policy * 3]:
    logged_ask(prompt, temperature=0, max_tokens=60)
pd.DataFrame(LOG)

,chars,in,out,secs
0,7,28,3,0.16
1,423,126,31,0.31
2,2519,605,25,0.48


Longer inputs take longer, but far less than proportionally: reading the prompt is fast compared with writing the answer, which happens one token at a time. Output length drives latency much more than input length.

## Exercise 3

Depends on what you have. Typical findings: a larger model gives better answers and is slower; hosted APIs have network latency but no loading time; token counts differ between models because tokenizers and chat templates differ.

## Exercise 4

1. **Hard-coded key.** Anyone who sees the notebook can spend your money. Use `.env` and `anthropic.Anthropic()`.
2. **Long instructions in the user message, repeated every time.** Put them in `system=`; consider prompt caching.
3. **`max_tokens=16000` for a one-word label** is harmless to the bill (you pay for tokens produced, not the limit), but a structured-output schema is the real fix: it guarantees a valid label.
4. **`r.content[0].text`** assumes the first block is text. Current models can return thinking blocks first. Join the text blocks.
5. **No error handling.** One rate-limit error stops the whole loop and loses the results so far. Retry with backoff and save progress.
6. **Customer text pasted straight after instructions.** Delimit it (09.8).

In [7]:
FIXED = """
import anthropic, json
client = anthropic.Anthropic()                      # key from .env / environment
SCHEMA = {"type": "object", "properties": {"category": {"type": "string", "enum": LABELS}},
          "required": ["category"], "additionalProperties": False}
for email in emails:
    r = client.messages.create(
        model="claude-opus-5-5", max_tokens=16000, system=LONG_INSTRUCTIONS,
        messages=[{"role": "user", "content": f"<email>{email}</email>"}],
        output_config={"format": {"type": "json_schema", "schema": SCHEMA}})
    text = "".join(b.text for b in r.content if b.type == "text")
    category = json.loads(text)["category"]
"""
print(FIXED)


import anthropic, json
client = anthropic.Anthropic()                      # key from .env / environment
SCHEMA = {"type": "object", "properties": {"category": {"type": "string", "enum": LABELS}},
          "required": ["category"], "additionalProperties": False}
for email in emails:
    r = client.messages.create(
        model="claude-opus-5-5", max_tokens=16000, system=LONG_INSTRUCTIONS,
        messages=[{"role": "user", "content": f"<email>{email}</email>"}],
        output_config={"format": {"type": "json_schema", "schema": SCHEMA}})
    text = "".join(b.text for b in r.content if b.type == "text")
    category = json.loads(text)["category"]

